In [ ]:
#necessary imports
import torch
import torchvision
import torch.nn as nn
import clip
from torch.nn import functional as F
from clip.simple_tokenizer import SimpleTokenizer as _Tokenizer
from tqdm import tqdm
from collections import OrderedDict
import itertools
import collections
from typing import List
import random


We will now create functions to correctly get our data and split it into base and novel classes.

In [ ]:
def get_data(data_dir="./data", transform=None):
    """Load Flowers102 train, validation and test sets.
    Args:
        data_dir (str): Directory where the dataset will be stored.
        transform (torch.Compose)
    Returns:
        tuple: A tuple containing the train, validation, and test sets.
    """
    train = torchvision.datasets.Flowers102(root=data_dir, split="train", download=True, transform=transform)
    val = torchvision.datasets.Flowers102(root=data_dir, split="val", download=True, transform=transform)
    test = torchvision.datasets.Flowers102(root=data_dir, split="test", download=True, transform=transform)
    return train, val, test

In [ ]:
def base_novel_categories(dataset):
    # set returns the unique set of all dataset classes
    all_classes = set(dataset._labels)
    # and let's count them
    num_classes = len(all_classes)

    # here list(range(num_classes)) returns a list from 0 to num_classes - 1
    # then we slice the list in half and generate base and novel category lists
    base_classes = list(range(num_classes))[:num_classes//2]
    novel_classes = list(range(num_classes))[num_classes//2:]
    return base_classes, novel_classes

In [ ]:
_, _, tmp_test = get_data()
base_classes, novel_classes = base_novel_categories(tmp_test)
CLASS_NAMES = ["pink primrose", "hard-leaved pocket orchid", "canterbury bells", "sweet pea",
                "english marigold", "tiger lily", "moon orchid", "bird of paradise", "monkshood",
                "globe thistle", "snapdragon", "colt's foot", "king protea", "spear thistle",
                "yellow iris", "globe-flower", "purple coneflower", "   ", "balloon flower",
                "giant white arum lily", "fire lily", "pincushion flower", "fritillary", "red ginger",
                "grape hyacinth", "corn poppy", "prince of wales feathers", "stemless gentian", "artichoke",
                "sweet william", "carnation", "garden phlox", "love in the mist", "mexican aster",
                "alpine sea holly", "ruby-lipped cattleya", "cape flower", "great masterwort", "siam tulip",
                "lenten rose", "barbeton daisy", "daffodil", "sword lily", "poinsettia", "bolero deep blue",
                "wallflower", "marigold", "buttercup", "oxeye daisy", "common dandelion", "petunia", "wild pansy",
                "primula", "sunflower", "pelargonium", "bishop of llandaff", "gaura", "geranium", "orange dahlia",
                "pink-yellow dahlia", "cautleya spicata", "japanese anemone", "black-eyed susan", "silverbush",
                "californian poppy", "osteospermum", "spring crocus", "bearded iris", "windflower", "tree poppy",
                "gazania", "azalea", "water lily", "rose", "thorn apple", "morning glory", "passion flower", "lotus",
                "toad lily", "anthurium", "frangipani", "clematis", "hibiscus", "columbine", "desert-rose",
                "tree mallow", "magnolia", "cyclamen", "watercress", "canna lily", "hippeastrum", "bee balm",
                "ball moss", "foxglove", "bougainvillea", "camellia", "mallow", "mexican petunia", "bromelia",
                "blanket flower", "trumpet creeper", "blackberry lily"]
print("Base Class Names:", [(i, CLASS_NAMES[i]) for i in base_classes])
print("Novel Class Names:", [(i, CLASS_NAMES[i]) for i in novel_classes])

Base Class Names: [(0, 'pink primrose'), (1, 'hard-leaved pocket orchid'), (2, 'canterbury bells'), (3, 'sweet pea'), (4, 'english marigold'), (5, 'tiger lily'), (6, 'moon orchid'), (7, 'bird of paradise'), (8, 'monkshood'), (9, 'globe thistle'), (10, 'snapdragon'), (11, "colt's foot"), (12, 'king protea'), (13, 'spear thistle'), (14, 'yellow iris'), (15, 'globe-flower'), (16, 'purple coneflower'), (17, '   '), (18, 'balloon flower'), (19, 'giant white arum lily'), (20, 'fire lily'), (21, 'pincushion flower'), (22, 'fritillary'), (23, 'red ginger'), (24, 'grape hyacinth'), (25, 'corn poppy'), (26, 'prince of wales feathers'), (27, 'stemless gentian'), (28, 'artichoke'), (29, 'sweet william'), (30, 'carnation'), (31, 'garden phlox'), (32, 'love in the mist'), (33, 'mexican aster'), (34, 'alpine sea holly'), (35, 'ruby-lipped cattleya'), (36, 'cape flower'), (37, 'great masterwort'), (38, 'siam tulip'), (39, 'lenten rose'), (40, 'barbeton daisy'), (41, 'daffodil'), (42, 'sword lily'), (4

Let's now split the dataset.

In [ ]:
def split_data(dataset, base_classes):
    # these two lists will store the sample indexes
    base_categories_samples = []
    novel_categories_samples = []

    # we create a set of base classes to compute the test below in O(1)
    # this is optional and can be removed
    base_set = set(base_classes)

    # here we iterate over sample labels and also get the correspondent sample index
    for sample_id, label in enumerate(dataset._labels):
        if label in base_set:
            base_categories_samples.append(sample_id)
        else:
            novel_categories_samples.append(sample_id)

    # here we create the dataset subsets
    # the torch Subset is just a wrapper around the dataset
    # it simply stores the subset indexes and the original dataset (your_subset.dataset)
    # when asking for sample i in the subset, torch will look for its original position in the dataset and retrieve it
    # https://pytorch.org/docs/stable/data.html#torch.utils.data.Subset
    base_dataset = torch.utils.data.Subset(dataset, base_categories_samples)
    novel_dataset = torch.utils.data.Subset(dataset, novel_categories_samples)
    return base_dataset, novel_dataset

In [ ]:
def create_remapped_dataset(dataset, selected_classes):
    """Create a dataset subset with remapped labels.

    Args:
        dataset: Original dataset
        selected_classes: List of class indices to include

    Returns:
        Subset dataset with labels remapped to [0, len(selected_classes)-1]
    """
    # Create mapping from original labels to new labels
    label_map = {old_label: new_label for new_label, old_label in enumerate(selected_classes)}
    selected_set = set(selected_classes)

    # Find samples and create new labels
    selected_samples = []
    new_labels = []

    for sample_id, label in enumerate(dataset._labels):
        if label in selected_set:
            selected_samples.append(sample_id)
            new_labels.append(label_map[label])

    # Create subset
    subset = torch.utils.data.Subset(dataset, selected_samples)

    # Add remapped labels to subset
    subset.remapped_labels = new_labels

    return subset

class RemappedDataset(torch.utils.data.Dataset):
    """Wrapper dataset that returns remapped labels"""
    def __init__(self, subset_dataset):
        self.dataset = subset_dataset.dataset
        self.indices = subset_dataset.indices
        self.labels = subset_dataset.remapped_labels

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        # Get original sample
        original_idx = self.indices[idx]
        image, _ = self.dataset[original_idx]  # Ignore original label

        # Return with remapped label
        return image, self.labels[idx]

Let's now load a pretrained CLIP model.

In [ ]:
device = "cuda"
print(device)

# Load the CLIP model and preprocessing transform
clip_model, preprocess = clip.load("ViT-B/16", device=device)

# Cast the model to float32 if on GPU to prevent dtype errors
if device == "cuda":
    clip_model.float()

cuda


Let's prepare the train-test-splits.


In [ ]:
# get the three datasets
train_set, val_set, test_set = get_data(transform=preprocess)

# split classes into base and novel
base_classes, novel_classes = base_novel_categories(train_set)

# split the three datasets
train_base, _ = split_data(train_set, base_classes)
val_base, _ = split_data(val_set, base_classes)
test_base, test_novel = split_data(test_set, base_classes)

In [ ]:
# MoCoOp: Mixture of Prompt Learning
HARD_GROUPS = [
    ["a photo of a type of flower: {}", "a photo of a type of flower: the {}."],    # flowers
    ["a photo of a {}.", "a photo of the {}."],                                     # generic
    ["a close-up photo of a {}.", "a macro photo of a {}."],                        # proximity
    ["a cropped photo of a {}.", "a cropped photo of the {}."],                     # crops
    ["a bright photo of a {}.", "a bright photo of the {}."],                       # brightness
    ["a good photo of a {}.", "a good quality photo of a {}."],                     # good quality
    ["a low resolution photo of a {}.", "a pixelated photo of a {}."],              # low resolution
    ["itap of a {}.", "itap of the {}."]                                            # I took a picture of ...
]

Now we will start implementing MoCoOp.


In [ ]:
class TextEncoder(nn.Module):
    def __init__(self, clip_model):
        super().__init__()
        self.transformer = clip_model.transformer
        self.positional_embedding = clip_model.positional_embedding
        self.ln_final = clip_model.ln_final
        self.text_projection = clip_model.text_projection
        self.dtype = clip_model.dtype

    def forward(self, prompts, tokenized_prompts):
        x = prompts + self.positional_embedding.type(self.dtype)
        x = x.permute(1, 0, 2)  # NLD -> LND
        x = self.transformer(x)
        x = x.permute(1, 0, 2)  # LND -> NLD
        x = self.ln_final(x).type(self.dtype)
        x = x[torch.arange(x.shape[0]), tokenized_prompts.argmax(dim=-1)] @ self.text_projection
        return x

In [ ]:
def moe_contrastive_loss(visual_features, class_prototypes, labels=None, t=0.07, instance_loss=False):
    # print(visual_features.shape, class_prototypes.shape, transpose(class_prototypes).shape)
    if not instance_loss:
        grouped_prototypes = class_prototypes
        b = visual_features.size(0)
        c = visual_features.size(1)
        d = visual_features.size(2)
        e = class_prototypes[0].size(0)
        final_logits = torch.einsum('gcd, ged->gce', visual_features, grouped_prototypes)
        logits = final_logits.view(-1, c, e)

    else:
        logits = t.exp() * torch.einsum('bd, bcd->bc', visual_features, class_prototypes)
    if labels is not None:
        labels = labels.reshape(-1)
        if not instance_loss:
            return F.cross_entropy(logits.reshape(len(labels), -1), labels), logits, None
        else:
            return F.cross_entropy(logits, labels), logits
    else:
        return None, logits

In [ ]:
_tokenizer = _Tokenizer()


class PromptLearner(nn.Module):
    def __init__(
            self,
            classnames,
            clip_model,
            text_encoder_model,
            all_classnames,
            device: torch.device,
            include_all_classes: bool = False,
            groups: int | List[int] = 0, # [2, 2, 2, 2, 2, 2, 2, 2]
            prompts: List[str] = [],
            enable_correction: bool = False,
            train_w: bool = False,
            use_base_subsample_classes: bool = False,
            use_mocoop: bool = True,
            ):
        super().__init__()
        self.device = device
        n_cls = len(all_classnames) if include_all_classes else len(classnames)

        cumulative_sum = [0] + list(itertools.accumulate(groups))[:-1]
        group_first_context_indices = cumulative_sum
        self.group_first_context_indices = group_first_context_indices
        self.class_token_position = []
        self.groups = groups
        dtype = clip_model.dtype
        ctx_dim = clip_model.ln_final.weight.shape[0]
        self.ctx_dim = ctx_dim
        vis_dim = clip_model.visual.output_dim
            
        self.n_context_tokens_before_classname = []
        self.n_context_tokens_after_classname = []
        all_prompt_templates_before_classname = [prompt.replace('.','').split('{}')[0][:-1] for prompt in prompts]
        all_prompt_templates_after_classname= [prompt.replace('.', '').split('{}')[1] for prompt in prompts]
        # use given words to initialize context vectors
        self.num_experts = len(group_first_context_indices)
        for i, idx in enumerate(group_first_context_indices):
            if len(all_prompt_templates_before_classname[idx]) == 0:
                n_ctx = 0
            else:
                n_ctx = len(all_prompt_templates_before_classname[idx].split(" "))
            prompt_template_before = all_prompt_templates_before_classname[idx].replace("_", " ")
            prompt = clip.tokenize(prompt_template_before).to(self.device)
            with torch.no_grad():
                embedding = (clip_model.token_embedding(prompt).type(dtype)).to(self.device)
            ctx_vectors_before = embedding[0, 1 : 1 + n_ctx, :] # exclude sos, eos
            self.register_parameter(f'ctx_before_{i}', nn.Parameter(ctx_vectors_before))
            self.n_context_tokens_before_classname.append(n_ctx)
            if len(all_prompt_templates_after_classname[idx]) == 0:
                n_ctx = 0
            else:
                n_ctx = len(all_prompt_templates_after_classname[idx].split(" "))
            prompt_template_after = all_prompt_templates_after_classname[idx].replace("_", " ")
            prompt = clip.tokenize(prompt_template_after).to(self.device)
            with torch.no_grad():
                embedding = (clip_model.token_embedding(prompt).type(dtype)).to(self.device)
            ctx_vectors_after = embedding[0, 1 : 1 + n_ctx, :] # exclude sos, eos
            self.register_parameter(f'ctx_after_{i}', nn.Parameter(ctx_vectors_after))
            self.n_context_tokens_after_classname.append(n_ctx)

        classnames = [name.replace("_", " ") for name in classnames]
        all_classnames = [name.replace("_", " ") for name in all_classnames]

        if include_all_classes:
            # Preserve class order
            classes_delta = [name for name in all_classnames if name not in classnames]
            print(f'Number of extra class names: {len(classes_delta)}')
            classnames += classes_delta
            print(f'Number of class names after: {len(classnames)}')
        
        name_token_lengths = [len(_tokenizer.encode(name)) for name in classnames]
        all_name_token_lengths = [len(_tokenizer.encode(name)) for name in all_classnames]
        
        self.tokenized_main_group_prompts_classes = []
        for i in range(self.num_experts):
            prompts = [all_prompt_templates_before_classname[group_first_context_indices[i]] + ' ' + name + all_prompt_templates_after_classname[group_first_context_indices[i]] for name in classnames]
            tokenized_prompts = (torch.cat([clip.tokenize(p) for p in prompts])).to(self.device)  # (n_cls, n_tkn)
            self.tokenized_main_group_prompts_classes.append(tokenized_prompts)
            with torch.no_grad():
                embedding = clip_model.token_embedding(tokenized_prompts).type(dtype).to(self.device)
                # These token vectors will be saved when in save_model(),
                # but they should be ignored in load_model() as we want to use
                # those computed using the current class names
            self.register_buffer(f"token_prefix_{i}", embedding[:, :1, :])  # SOS
            self.register_buffer(f"token_suffix_{i}", embedding[:, 1 + self.n_context_tokens_before_classname[i] :, :])  # CLS, EOS

        if use_base_subsample_classes:
            self.construct_references_lasp(
                clip_model=clip_model,
                text_encoder_model=text_encoder_model,
                all_classnames=all_classnames,
                prompt_prefixs=all_prompt_templates_before_classname,
                prompt_suffixs=all_prompt_templates_after_classname,
                dtype=dtype)
        else:
            self.construct_references_lasp(
                clip_model=clip_model,
                text_encoder_model=text_encoder_model,
                all_classnames=classnames,
                prompt_prefixs=all_prompt_templates_before_classname,
                prompt_suffixs=all_prompt_templates_after_classname,
                dtype=dtype)

        self.gate = nn.Linear(vis_dim, self.num_experts)

        self.n_active_classes = n_cls
        self.name_token_lengths = name_token_lengths
        self.all_name_token_lengths = all_name_token_lengths
        self.all_classnames = all_classnames
        self.classnames = classnames

        if enable_correction:
            self.w = nn.Parameter(torch.zeros(1, ctx_dim, device=embedding.device, dtype=dtype), requires_grad=train_w)
        
    def reset_classnames(self, classnames, clip_model, prompts, ctx_init, dtype, use_base_subsample_classes, text_encoder_model, all_classnames):
        # Update classnames and name_lens
        self.classnames = [name.replace("_", " ") for name in classnames]
        self.all_classnames = [name.replace("_", " ") for name in all_classnames]
        self.n_active_classes = len(self.classnames)
        self.name_token_lengths = [len(_tokenizer.encode(name)) for name in self.classnames]
        self.all_name_token_lengths = [len(_tokenizer.encode(name)) for name in all_classnames]

        prompt_template_before_all = [prompt.replace('.', '').split('{}')[0][:-1] for prompt in prompts]
        prompt_template_after_all = [prompt.replace('.', '').split('{}')[1] for prompt in prompts]

        self.tokenized_main_group_prompts_classes = []
        for i in range(self.num_experts):
            prompts_i = [prompt_template_before_all[ctx_init[i]] + ' ' + name + prompt_template_after_all[ctx_init[i]] for name in self.classnames]
            tokenized_prompts = (torch.cat([clip.tokenize(p) for p in prompts_i])).to(self.device)
            self.tokenized_main_group_prompts_classes.append(tokenized_prompts)
            with torch.no_grad():
                embedding = clip_model.token_embedding(tokenized_prompts).type(dtype).to(self.device)
            self.register_buffer(f"token_prefix_{i}", embedding[:, :1, :])
            self.register_buffer(f"token_suffix_{i}", embedding[:, 1 + self.n_context_tokens_before_classname[i] :, :])
        if use_base_subsample_classes:
            self.construct_references_lasp(clip_model, text_encoder_model, all_classnames, prompt_template_before_all, prompt_template_after_all, dtype)
        else:
            self.construct_references_lasp(clip_model, text_encoder_model, classnames, prompt_template_before_all, prompt_template_after_all, dtype)

    def construct_references_lasp(self, clip_model, text_encoder_model, all_classnames, prompt_prefixs, prompt_suffixs, dtype):
        """Saves a tensor with the text features of all possible combinations of prompts and class names, and a tensor with the average text features for each group,
        averaged across group and classes."""
        # template_prompts = cfg.TRAINER.MoCoOp.CTX_INIT
        all_classnames = [name.replace("_", " ") for name in all_classnames]

        all_class_full_prompt_text_features = []
        for i in range(len(prompt_prefixs)):
            prompts = [prompt_prefixs[i] + ' ' +  name + prompt_suffixs[i] + '.' for name in all_classnames]
            # prompts = [c_init + " " + name + "." for name in all_classnames]
            tokenized_prompt_all_classes = (torch.cat([clip.tokenize(p) for p in prompts])).to(self.device)  # (n_cls, n_tkn)
            text_encoder_model.to(self.device)
            # text_encoder_model.cuda()
            with torch.no_grad():
                embedding_all_classes = (clip_model.token_embedding(tokenized_prompt_all_classes).cuda().type(dtype)).to(self.device)
                class_full_prompt_text_features = text_encoder_model(embedding_all_classes, tokenized_prompt_all_classes).type(dtype)
                all_class_full_prompt_text_features.append(class_full_prompt_text_features)
            
        class_full_prompt_text_features = torch.stack(all_class_full_prompt_text_features, dim=0)
        class_full_prompt_text_features = class_full_prompt_text_features / class_full_prompt_text_features.norm(dim=-1, keepdim=True)
        self.register_buffer("class_text_features", class_full_prompt_text_features)
        grouped_prototypes = torch.stack([group.mean(dim=0) for group in torch.split(class_full_prompt_text_features, self.groups)])
        grouped_prototypes = grouped_prototypes / grouped_prototypes.norm(dim=-1, keepdim=True)
        self.register_buffer("grouped_prototypes", grouped_prototypes)

        self.tokenized_main_group_prompts_all_classes = []
        for i in range(self.num_experts):
            prompts = [prompt_prefixs[self.group_first_context_indices[i]] + " " + name + prompt_suffixs[self.group_first_context_indices[i]] + '.' for name in all_classnames]
            tokenized_main_group_prompt_all_classes = (torch.cat([clip.tokenize(p) for p in prompts])).to(self.device)  # (n_cls, n_tkn)
            self.tokenized_main_group_prompts_all_classes.append(tokenized_main_group_prompt_all_classes)
            with torch.no_grad():
                embedding = clip_model.token_embedding(tokenized_main_group_prompt_all_classes).type(dtype).to(self.device)

            self.register_buffer(f"token_prefix_all_{i}", embedding[:, :1, :])  # SOS
            self.register_buffer(f"token_suffix_all_{i}", embedding[:, 1 + self.n_context_tokens_before_classname[i]:, :])  # CLS, EOS

        # self.ref_tokenized_prompts_all = tokenized_prompts_all_c
        self.n_all_classes = len(prompts)
    
    def construct_prompts(self, n_ctx_after, ctx_before, ctx_after, prefix, suffix, name_lengths, label=None):
        """Returns list of tensors of shape (n_cls, n_token, token_dim)
        where n_token is the length of the following list
        [SOS]
        [Learnable prefix token 1]
        ...
        [Learnable prefix token m]
        [Class token 1]
        ...
        [Class token n]
        [Learnable suffix token 1]
        ...
        [Learnable suffix token o]
        [SOS]
        """
        # dim0 is either batch_size (during training) or n_cls (during testing)
        # ctx: context tokens, with shape of (dim0, n_ctx, ctx_dim)
        # prefix: the sos token, with shape of (n_cls, 1, ctx_dim)
        # suffix: remaining tokens, with shape of (n_cls, *, ctx_dim)

        if label is not None:
            prefix = prefix[label]
            suffix = suffix[label]
        
       
        prompts = []
        for i in range(len(prefix)):
            name_length = name_lengths[i]
            prefix_i = prefix[i : i + 1, :, :]
            class_i = suffix[i : i + 1, :name_length, :]
            # print(i, len(prefix), len(suffix), len(suffix[i]), name_len+self.n_ctx_after[i])
            suffix_i = suffix[i : i + 1, name_length+n_ctx_after:, :]
            # print(ctx_before.shape)
            # print(prefix_i.shape, ctx_before.shape, class_i.shape, ctx_after.shape, suffix_i.shape)
            prompt = torch.cat(
                [
                    prefix_i,     # (1, 1, dim)
                    ctx_before[i].unsqueeze(0),  # (1, n_ctx//2, dim)
                    class_i,      # (1, name_len, dim)
                    ctx_after[i].unsqueeze(0),  # (1, n_ctx//2, dim)
                    suffix_i,     # (1, *, dim)
                ],
                dim=1,
            )
            prompts.append(prompt)
        prompts = torch.cat(prompts, dim=0)

       
        return prompts

 
    
    
    def forward(self, all=False):
        prompts = []
        for i in range(self.num_experts):
            ctx_before = getattr(self, f'ctx_before_{i}')# tokens before class name - SOS
            ctx_after = getattr(self, f'ctx_after_{i}')# tokens after class name - EOS
            # ctx_all_experts.append(ctx)
            # Use instance-conditioned context tokens for all classes
            if not all:
                prefix = getattr(self, f"token_prefix_{i}")# embedding SOS
                suffix = getattr(self, f"token_suffix_{i}")# embeddings "{class_name}.[EOS]"
                n_cls = self.n_active_classes
                name_token_lengths = self.name_token_lengths
            else:
                prefix = getattr(self, f"token_prefix_all_{i}")# embedding SOS on all classes
                suffix = getattr(self, f"token_suffix_all_{i}")# embeddings "{class_name}.[EOS]" on all classes
                n_cls = len(self.all_classnames)
                name_token_lengths = self.all_name_token_lengths  
            ctx_before_i = ctx_before.unsqueeze(0).expand(n_cls, -1, -1)
            ctx_after_i = ctx_after.unsqueeze(0).expand(n_cls, -1, -1)
            pts_i = self.construct_prompts(
                n_ctx_after=self.n_context_tokens_after_classname[i],
                ctx_before=ctx_before_i,
                ctx_after=ctx_after_i,
                prefix=prefix,
                suffix=suffix,
                name_lengths=name_token_lengths
            )  # (n_cls, n_tkn, ctx_dim)
            prompts.append(pts_i)

        return prompts

In [ ]:
def increase_top2_logits(logits, increment=1.0):
    _, top2_indices = torch.topk(logits, k=2)
    increment_tensor = torch.zeros_like(logits)
    increment_tensor.scatter_(1, top2_indices, increment)

    logits += increment_tensor
    return logits


class CustomCLIP(nn.Module):
    def __init__(
            self,
            classnames,
            clip_model,
            all_classnames,
            device,
            include_all_classes: bool,
            groups: int | List[int],
            prompts: List[str],
            enable_correction: bool,
            train_w: bool,
            use_base_subsample_classes: bool,
            use_mocoop: bool,
            train_batch_size: int,
            text_loss_weight: float = 1.0,
            gate_loss_weight: float = 1.0
            ):
        super().__init__()
        self.train_batch_size = train_batch_size
        self.text_loss_weight = text_loss_weight
        self.gate_loss_weight = gate_loss_weight
        self.use_mocoop = use_mocoop
        self.image_encoder = clip_model.visual
        self.enable_correction = enable_correction
        self.use_base_subsample_classes = use_base_subsample_classes
        self.device = device
        # self.text_encoder = TextEncoder(clip_model)
        self.text_encoder = nn.DataParallel(TextEncoder(clip_model))
        self.prompt_learner = PromptLearner(
            classnames=classnames,
            clip_model=clip_model,
            text_encoder_model=self.text_encoder,
            all_classnames=all_classnames,
            device=self.device,
            include_all_classes=include_all_classes,
            groups=groups,
            prompts=prompts,
            enable_correction=enable_correction,
            train_w=train_w,
            use_base_subsample_classes=use_base_subsample_classes,
            use_mocoop=use_mocoop
            ).to(self.device)
        self.prompt_learner.gate = self.prompt_learner.gate.to(self.device)
        self.tokenized_main_group_prompts_classes = self.prompt_learner.tokenized_main_group_prompts_classes
        self.grouped_prototypes = self.prompt_learner.grouped_prototypes
        self.group_features = self.grouped_prototypes
        # self.group_features = self.grouped_prototypes.mean(dim=1)
        self.group_features = self.group_features / self.group_features.norm(dim=-1, keepdim=True)
        if use_mocoop:
            self.tokenized_main_group_prompts_all_classes = self.prompt_learner.tokenized_main_group_prompts_all_classes
            self.num_templates = len(self.prompt_learner.class_text_features)
        self.logit_scale = clip_model.logit_scale
        self.dtype = clip_model.dtype
        self.n_all_classnames = len(all_classnames)
        self.n_classnames = len(classnames)
        self.num_experts = self.prompt_learner.num_experts
        # self.loss = contrastive_loss
        self.loss = moe_contrastive_loss
        self.gate_count = collections.defaultdict(int)
        self.eps = 0.1
        self.max_iter = 100
        self.groups = groups
        # self.dataset = dataset


    def compute_gated_text_feature(
            self,
            image_features,
            tokenized_prompts_all,# contains tokenized prompts for all experts
            text_gating=None
            ):
        expert_text_features_all = []
        if text_gating is None:
            gating_distribution = (self.prompt_learner.gate(image_features)).to(self.device)
        else:
            gating_distribution = text_gating
        topk_gating_distribution, indices = torch.topk(gating_distribution, k=2)
        prompts = self.prompt_learner()# get prompts embeddings for all experts
        
        for i, prompt in enumerate(prompts):# cycle over main prompt groups
            if self.train_batch_size < 4 and i not in indices:
                text_features_per_expert = torch.zeros(
                    self.n_classnames,
                    self.text_encoder.module.ln_final.weight.shape[0],
                    device=image_features.device,
                    dtype=image_features.dtype
                    )
            else:
                tokenized_prompts = tokenized_prompts_all[i]
                text_features_per_expert = self.text_encoder(prompt, tokenized_prompts)
                if self.enable_correction:
                    w = self.prompt_learner.w
                    text_features_per_expert = text_features_per_expert + w
                text_features_per_expert = text_features_per_expert / text_features_per_expert.norm(dim=-1, keepdim=True)
            expert_text_features_all.append(text_features_per_expert)
        text_features_all = []
        for batch_idx in range(len(image_features)):
            text_features = []
            for i in range(len(indices[batch_idx])):
                text_features.append(expert_text_features_all[indices[batch_idx][i]])
            text_features = torch.stack(text_features)
            text_features_all.append(text_features)
            
        topk_gating_distribution = torch.softmax(topk_gating_distribution, dim=1)
        # print(topk_gating_distribution.shape, torch.stack(text_features_all).shape)
        text_features = torch.einsum("bk,bkcd->bcd", topk_gating_distribution, torch.stack(text_features_all))
        return text_features, gating_distribution, indices, torch.stack(expert_text_features_all)
        # return text_features_per_expert, gating_distribution

    def forward_text_to_text(self, text_features=None, indices=None):
        with torch.no_grad():
            class_text_features = self.prompt_learner.class_text_features
            # class_text_features = class_text_features / class_text_features.norm(dim=-1, keepdim=True)

        if torch.rand(1).item() < 0.5:
            noise = 0.05 * torch.randn_like(class_text_features)
            class_text_features.add_(noise)
        # """
        if self.use_base_subsample_classes:
            prompts = self.prompt_learner(all=True)
            expert_text_features_all = []
            for i, prompt in enumerate(prompts):
                if self.train_batch_size < 4 and i not in indices:
                    text_features_per_expert = torch.zeros(self.n_all_classnames, self.text_encoder.module.ln_final.weight.shape[0], device=self.device, dtype=self.dtype)
                else:
                    tokenized_prompts = self.tokenized_main_group_prompts_all_classes[i]
                    # print(self.text_encoder.module.ln_final.weight.device, prompt.device)
                    text_features_per_expert = self.text_encoder(prompt, tokenized_prompts)

                    if self.enable_correction:
                        w = self.prompt_learner.w
                        # w = self.prompt_learner.w[i].unsqueeze(0)
                        # print(text_features_per_expert.shape, w.shape)
                        if len(text_features_per_expert.shape) == 2:
                            text_features_per_expert = text_features_per_expert + w
                        elif len(text_features_per_expert.shape) == 3:
                            text_features_per_expert = text_features_per_expert + w.unsqueeze(0)
                    

                    text_features_per_expert = text_features_per_expert / text_features_per_expert.norm(dim=-1, keepdim=True)
                expert_text_features_all.append(text_features_per_expert)
            text_features = torch.stack(expert_text_features_all)
        # else:
            # assert self.prompt_learner.all_classnames == self.prompt_learner.classnames
        # """
        # text_features = text_features.unsqueeze(0)

        label = torch.arange(self.prompt_learner.n_all_classes, device=class_text_features.device, dtype=torch.long).unsqueeze(0).expand(len(text_features), -1)

        # return sum([self.loss(text_features, class_text_features, label, t=self.logit_scale)[0] for text_features in expert_text_features_all])/len(expert_text_features_all)
        # label = torch.arange(self.prompt_learner.n_cls_all, device=class_text_features.device, dtype=torch.long).unsqueeze(0).expand(2, -1)
     
        # print(text_features.shape, self.grouped_prototypes.shape)
        assert text_features.shape == self.grouped_prototypes.shape
        loss, _, group_features = self.loss(text_features, self.grouped_prototypes, label, t=self.logit_scale)
        return loss, group_features
    
    

    def forward(self, image, label=None):

        image_features = self.image_encoder(image.type(self.dtype))
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        tokenized_prompts_all_experts = self.tokenized_main_group_prompts_classes

        text_features, gating_distribution, indices, expert_text_features = self.compute_gated_text_feature(image_features, tokenized_prompts_all_experts)
        if self.use_mocoop:
            p = 1.
            if random.random() < p:
                loss_text, group_features = self.forward_text_to_text(expert_text_features, indices=indices)
            else:
                loss_text = 0
            # group_features = F.normalize(group_features, dim=-1)
            if label is not None:
                group_features = self.group_features[:, label]
                text_gating = torch.einsum('bd, gbd->bg', image_features, group_features)
                text_gating = increase_top2_logits(text_gating, increment=10.0)
        loss, logits = self.loss(image_features, text_features, label, t=self.logit_scale, instance_loss=True)

        if self.prompt_learner.training:
            if self.use_mocoop:
                loss += self.text_loss_weight * loss_text / p

        if self.prompt_learner.training:
            if self.use_mocoop:
                return loss  + self.gate_loss_weight * F.cross_entropy(gating_distribution, F.softmax(text_gating, dim=1)), logits
            else:
                return loss, logits
        else:
            return logits
        
    def save(self, path):
        torch.save({"prompt_state": self.prompt_learner.state_dict()}, path)

    def load(self, path, strict=True, map_location="cpu"):
        ckpt = torch.load(path, map_location=map_location)
        self.prompt_learner.load_state_dict(ckpt["prompt_state"], strict=strict)

Training and eval


In [ ]:
# Remap base classes
train_base_remapped = create_remapped_dataset(train_set, base_classes)
val_base_remapped   = create_remapped_dataset(val_set, base_classes)
test_base_remapped  = create_remapped_dataset(test_set, base_classes)

train_base_dataset = RemappedDataset(train_base_remapped)
val_base_dataset   = RemappedDataset(val_base_remapped)
test_base_dataset  = RemappedDataset(test_base_remapped)

train_batch_size = 1
test_batch_size = 8
train_loader = torch.utils.data.DataLoader(train_base_dataset, batch_size=train_batch_size, shuffle=True,  num_workers=2)
val_loader   = torch.utils.data.DataLoader(val_base_dataset,   batch_size=test_batch_size, shuffle=False, num_workers=2)
test_loader  = torch.utils.data.DataLoader(test_base_dataset,  batch_size=test_batch_size, shuffle=False, num_workers=2)

In [ ]:
all_class_names = CLASS_NAMES
groups = [len(g) for g in HARD_GROUPS]
prompts = [p for group in HARD_GROUPS for p in group]
classnames = [all_class_names[i] for i in base_classes]
model = CustomCLIP(
    include_all_classes=False,
    groups=groups,
    prompts=prompts,
    enable_correction=False,
    train_w=False,
    use_base_subsample_classes=True,
    use_mocoop=True,
    train_batch_size=train_batch_size,
    text_loss_weight=5.0,
    gate_loss_weight=1.0,
    classnames=classnames,
    clip_model=clip_model,
    all_classnames=all_class_names,
    device=next(clip_model.parameters()).device
    )

print("Turning off gradients in both the image and the text encoder")
name_to_update = "prompt_learner"

for name, param in model.named_parameters():
    if name_to_update not in name:
        param.requires_grad_(False)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.002, momentum=0.9)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-5)

num_epochs = 10
best_val = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_acc': []}

Turning off gradients in both the image and the text encoder


In [ ]:
def evaluate_mocoop(model, loader, device):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)                # targets=None -> returns logits, aux
            pred = logits.argmax(dim=1)
            correct += (pred == labels).sum().item()
            total   += labels.numel()
    return correct / max(total, 1)

print(f"Trainable params: {sum(p.numel() for p in params):,}")

model.train()
best_val = 0

Trainable params: 24,584


In [ ]:
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        images = images.to(torch.float32) # Explicitly cast images to float32

        optimizer.zero_grad()
        loss, logits = model(images, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total   += images.size(0)

    scheduler.step()
    train_loss = running_loss / total
    train_acc  = correct / total
    val_acc    = evaluate_mocoop(model, val_loader, device)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    if val_acc > best_val:
        best_val = val_acc
        # Save only prompt state (your class exposes .save/.load)
        model.save('best_mocoop_prompt.pth')
    print(f"Epoch {epoch+1}/{num_epochs} | loss {train_loss:.4f} | acc {train_acc:.4f} | val {val_acc:.4f}")

print(f"Best val: {best_val:.4f}")

In [ ]:
# Load best prompt and test on base
model.load('best_mocoop_prompt.pth', strict=False, map_location=device)   # loads into model.prompt
base_test_acc = evaluate_mocoop(model, test_loader, device)
print(f"Base test acc: {base_test_acc:.4f}")

In [ ]:
# ===== Novel class evaluation =====
# Switch to novel classes (reuse same model, just change active classes)
novel_class_names = [CLASS_NAMES[i] for i in novel_classes]
novel_model = CustomCLIP(
    include_all_classes=False,
    groups=groups,
    prompts=prompts,
    enable_correction=False,
    train_w=False,
    use_base_subsample_classes=True,
    use_mocoop=True,
    train_batch_size=train_batch_size,
    text_loss_weight=5.0,
    gate_loss_weight=1.0,
    classnames=novel_class_names,
    clip_model=clip_model,
    all_classnames=all_class_names,
    device=next(clip_model.parameters()).device
)

novel_model.load('best_mocoop_prompt.pth', strict=False, map_location=device)
# PromptLearner.n_cls = len(novel_class_names)
# novel_model.prompt_learner.construct_references_lasp(
#     clip_model, novel_model.text_encoder.module, novel_class_names, 
#     [p.replace('.', '').split('{}')[0][:-1] for p in prompts], 
#     [p.replace('.', '').split('{}')[1] for p in prompts], 
#     clip_model.dtype
# )
novel_model.prompt_learner.reset_classnames(
    novel_class_names,
    clip_model,
    prompts,
    novel_model.prompt_learner.group_first_context_indices,
    clip_model.dtype,
    use_base_subsample_classes=True,
    text_encoder_model=novel_model.text_encoder.module,
    all_classnames=all_class_names
)

# Novel dataset and loader
test_novel_remapped = create_remapped_dataset(test_set, novel_classes)
test_novel_dataset  = RemappedDataset(test_novel_remapped)
test_novel_loader   = torch.utils.data.DataLoader(test_novel_dataset, batch_size=test_batch_size, shuffle=False, num_workers=2)

novel_test_acc = evaluate_mocoop(novel_model, test_novel_loader, device)
print(f"Novel test acc: {novel_test_acc:.4f}")
print(f"Harmonic mean: {(2 / (1/base_test_acc + 1/novel_test_acc)):.4f}")